In [1]:
START_DATE = "07/13/2020"
END_DATE = "04/22/2024"
TRAIN_SPLIT_IDX = 35
TIME_WINDOW_SIZE = 7
TEMPORAL_EDGE_WINDOW_SIZE = 1

In [2]:
RAW_HOSPITALZATION_FILE = "../data/raw/COVID-19_Reported_Patient_Impact_and_Hospital_Capacity_by_Facility_20251026.csv"
HOSP_COL = "total_adult_patients_hospitalized_confirmed_covid_7_day_sum"

In [3]:
import pandas as pd

from cgnn.process_xwalk import get_county_cbsa_map
from cgnn.utils import get_date_range, get_cbsa_list

In [4]:
max_missing_weeks=21
hosp_df = pd.read_csv(RAW_HOSPITALZATION_FILE, dtype={"fips_code": str})

hosp_df["collection_week"] = pd.to_datetime(hosp_df["collection_week"])
# Shift dates forward by one day to align with advan data
hosp_df["collection_week"] = hosp_df["collection_week"] + pd.Timedelta(days=1)


/local/ipykernel_388279/4172186702.py:2: DtypeWarning: Columns (0,3) have mixed types. Specify dtype option on import or set low_memory=False.
  hosp_df = pd.read_csv(RAW_HOSPITALZATION_FILE, dtype={"fips_code": str})


In [5]:
hosp_df.shape

(1045406, 128)

In [6]:
# filter  between START_DATE and END_DATE
hosp_df = hosp_df.loc[
    (hosp_df["collection_week"] >= START_DATE)
    & (hosp_df["collection_week"] <= END_DATE)
]

hosp_df.shape

(978984, 128)

In [7]:
# cutoff by max missing weeks
end_date = hosp_df["collection_week"].max()
dates = get_date_range(START_DATE, END_DATE)

total_weeks = len(dates)

# Calculate reporting statistics per hospital
hospital_reporting = (
    hosp_df.groupby("hospital_pk")
    .agg(
        {
            "collection_week": "nunique",
            "state": "first",
        }
    )
    .reset_index()
)
hospital_reporting.columns = ["hospital_pk", "unique_weeks", "state"]

# Calculate missing weeks
hospital_reporting["missing_weeks"] = (
    total_weeks - hospital_reporting["unique_weeks"]
)

In [8]:
hospital_reporting["missing_weeks"].min()

0

In [9]:
# filter out hospitals with more than MAX_MISSING_WEEKS missing weeks
hosp_df = hosp_df.merge(hospital_reporting, on="hospital_pk", how="left")
hosp_df = hosp_df.loc[hosp_df["missing_weeks"] <= max_missing_weeks]

hosp_df.shape

(833166, 131)

In [10]:
hosp_df[HOSP_COL] = hosp_df[HOSP_COL].replace("-999,999", 0)
hosp_df[HOSP_COL] = hosp_df[HOSP_COL].replace(-999999, 0)

hosp_df[HOSP_COL] = pd.to_numeric(hosp_df[HOSP_COL], errors="coerce")

hosp_df.shape

(833166, 131)

In [11]:
# merge in CBSA info and group by
county_cbsa_map = get_county_cbsa_map()

unique ZIPs in zip-cbsa map: 39502
unique ZIPs in zip-county map: 39490
num zips in cbsa but not county: 15
num zips in county but not cbsa: 3


In [12]:
print(len(set(hosp_df["fips_code"].unique())))
print(len(county_cbsa_map))

2174
3223


In [13]:
len(set(hosp_df["fips_code"].unique()) - set(county_cbsa_map["COUNTY"]))

14

In [14]:
# Check for duplicate keys that could cause row multiplication in merge
print("=== Checking for duplicate keys before merge ===\n")

# Check hosp_df for duplicate fips_code (expected - multiple hospitals per county)
hosp_fips_duplicates = hosp_df["fips_code"].value_counts()
print(f"hosp_df shape: {hosp_df.shape}")
print(f"Unique fips_code in hosp_df: {hosp_df['fips_code'].nunique()}")
print(f"Total rows in hosp_df: {len(hosp_df)}")
print(f"Expected rows if no duplicates: {hosp_df['fips_code'].nunique()}")
print(f"Multiple rows per fips_code: {len(hosp_df) > hosp_df['fips_code'].nunique()}\n")

# Check county_cbsa_map for duplicate COUNTY values (this would cause multiplication!)
county_duplicates = county_cbsa_map["COUNTY"].value_counts()
duplicate_counties = county_duplicates[county_duplicates > 1]

print(f"county_cbsa_map shape: {county_cbsa_map.shape}")
print(f"Unique COUNTY in county_cbsa_map: {county_cbsa_map['COUNTY'].nunique()}")
print(f"Total rows in county_cbsa_map: {len(county_cbsa_map)}")
print(f"Counties with duplicate entries: {len(duplicate_counties)}")

if len(duplicate_counties) > 0:
    print(f"\n⚠️  WARNING: Found {len(duplicate_counties)} counties with multiple CBSA mappings!")
    print("\nTop counties with multiple CBSA mappings:")
    print(duplicate_counties.head(10))
    
    print("\nExample of duplicate mappings:")
    example_county = duplicate_counties.index[0]
    print(f"\nCounty {example_county} maps to:")
    print(county_cbsa_map[county_cbsa_map["COUNTY"] == example_county][["COUNTY", "CBSA"]])
    
    # Calculate expected row multiplication
    print(f"\n📊 Impact analysis:")
    print(f"If a county has {duplicate_counties.max()} CBSA mappings, each hospital in that county")
    print(f"will create {duplicate_counties.max()} rows in the merged result.")
    
    # Show which counties in hosp_df would be affected
    affected_counties = set(hosp_df["fips_code"].unique()) & set(duplicate_counties.index)
    print(f"\nCounties in hosp_df that have multiple CBSA mappings: {len(affected_counties)}")
    if len(affected_counties) > 0:
        print("Sample affected counties:")
        for county in list(affected_counties)[:5]:
            hosp_count = (hosp_df["fips_code"] == county).sum()
            cbsa_count = duplicate_counties[county]
            print(f"  County {county}: {hosp_count} hospital rows × {cbsa_count} CBSA mappings = {hosp_count * cbsa_count} rows")
else:
    print("\n✓ No duplicate COUNTY values found in county_cbsa_map")

print("\n" + "="*60)


=== Checking for duplicate keys before merge ===

hosp_df shape: (833166, 131)
Unique fips_code in hosp_df: 2174
Total rows in hosp_df: 833166
Expected rows if no duplicates: 2174
Multiple rows per fips_code: True

county_cbsa_map shape: (3223, 2)
Unique COUNTY in county_cbsa_map: 3223
Total rows in county_cbsa_map: 3223
Counties with duplicate entries: 0

✓ No duplicate COUNTY values found in county_cbsa_map



In [51]:
county_cbsa_map.loc[county_cbsa_map["COUNTY"].str.startswith("02")].sort_values(by="COUNTY")

,COUNTY,CBSA
39259,02013,99999
39252,02016,99999
39225,02020,11260
39251,02050,99999
39305,02060,99999
39277,02063,99999
39270,02066,99999
39383,02068,99999
39261,02070,99999
39380,02090,21820


In [56]:
hosp_df.loc[hosp_df["fips_code"].isin(['02080',
        '02120',
        '02210',
        '02260',
        '02280']), ['fips_code','city']].drop_duplicates()

,fips_code,city
6256,02120,HOMER
6268,02280,PETERSBURG
6650,02120,SOLDOTNA
7074,02080,CORDOVA
7403,02210,SEWARD
8515,02260,VALDEZ


In [42]:
from typing import Any


missing_counties = list(set[Any](hosp_df["fips_code"].unique()) - set(county_cbsa_map["COUNTY"]))

sorted(missing_counties)

['02080',
 '02120',
 '02210',
 '02260',
 '02280',
 '09001',
 '09003',
 '09005',
 '09007',
 '09009',
 '09011',
 '09013',
 '09015',
 '51595']

In [30]:
hosp_df.merge(
    county_cbsa_map, left_on="fips_code", right_on="COUNTY", how="inner"
).shape

(833166, 133)

In [ ]:
hosp_df = hosp_df.groupby(["CBSA", "collection_week"])[HOSP_COL].sum().reset_index()
hosp_df = hosp_df.loc[hosp_df["CBSA"] != "99999"]

cbsa_list = get_cbsa_list()
hosp_df = hosp_df[hosp_df["CBSA"].isin(cbsa_list)]

hosp_df.shape

In [ ]:
# ensure all CBSAs have full date range and fill missing values with 0
multi_index = pd.MultiIndex.from_product(
    [hosp_df["CBSA"].unique(), dates], names=["CBSA", "collection_week"]
)
# Set index and reindex to fill missing combinations with 0
hosp_df = (
    hosp_df.set_index(["CBSA", "collection_week"])
    .reindex(multi_index, fill_value=0)
    .reset_index()
)

hosp_df.sort_values(by=["CBSA", "collection_week"], inplace=True)

# create node key
hosp_df["node_key"] = (
    hosp_df["CBSA"].astype(str)
    + "-"
    + hosp_df["collection_week"].dt.strftime("%Y-%m-%d")
)